# create database for later data analysis using sqlalchemy
Run this script once after model_data_extractor work is finished

REVAMP with intervall mapping for half life

In [1]:
import sys
from typing import List
import numpy as np
import pandas as pd
from pandas import read_csv
from pathlib import Path

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.rdkit_tools import smiles_standardizer_msc

In [2]:
# set directories and filenames, load data, create db if not exists
working_dir = Path.cwd().parent
data_dir = working_dir / "processed_data"

# create SQLAlchemy engine and session
database_file = data_dir / "hsbd_t_half_all.db"
engine = sa.create_engine(f"sqlite:///{database_file}")
Session = sessionmaker(bind=engine)
session = Session()

# drop all tables and recreate from scratch (full rebuild)
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# load data depending on whether the file with missing data included exists; if it does, load that, otherwise load the one without missing data; checking only for one file is sufficient since they are all generated together in the previous notebook
final_dataset = "incl" if (data_dir / f"hsbd_air_t_half_data_incl_missing.csv").exists() else "excl"
print(f"Loading data with missing values {'included' if final_dataset == 'incl' else 'excluded'}")
air_file = data_dir / f"hsbd_air_t_half_data_{final_dataset}_missing.csv"
soil_file = data_dir / f"hsbd_soil_t_half_data_{final_dataset}_missing.csv"
water_file = data_dir / f"hsbd_water_t_half_data_{final_dataset}_missing.csv"
sediment_file = data_dir / f"hsbd_sediment_t_half_data_{final_dataset}_missing.csv"
air_data_all = read_csv(air_file, index_col=0, header=0, sep="\t")
soil_data_all = read_csv(soil_file, index_col=0, header=0, sep="\t")
water_data_all = read_csv(water_file, index_col=0, header=0, sep="\t")
sediment_data_all = read_csv(sediment_file, index_col=0, header=0, sep="\t")

# insert into df counter of format "table_name"+<counter> with counter being 4 digits to each table called "reference"
air_data_all.insert(0, "reference", ["air_" + str(i).zfill(4) for i in range(len(air_data_all))])
soil_data_all.insert(0, "reference", ["soil_" + str(i).zfill(4) for i in range(len(soil_data_all))])
water_data_all.insert(0, "reference", ["water_" + str(i).zfill(4) for i in range(len(water_data_all))])
sediment_data_all.insert(0, "reference", ["sediment_" + str(i).zfill(4) for i in range(len(sediment_data_all))])

# simplify column names for half-life to T_half_days
air_data_all = air_data_all.rename(columns={"T_half_air_days": "T_half_days"})
soil_data_all = soil_data_all.rename(columns={"T_half_soil_days": "T_half_days"})
water_data_all = water_data_all.rename(columns={"T_half_water_days": "T_half_days"})
sediment_data_all = sediment_data_all.rename(columns={"T_half_sediment_days": "T_half_days"})

Loading data with missing values included


In [3]:
# --- Interval mapping for T_half_days ---
# Load the interval mapping table
interval_map_file = Path.cwd() / "hsbd_interval_mapping.csv"
interval_map = read_csv(interval_map_file)

# Sentinel value used in place of Infinity for upper bounds in the last bin
INFINITY_SENTINEL = 9_999_999.0

# Pre-process interval bounds: replace 'Infinity' strings with the sentinel float
interval_map["upper_bound_days"] = pd.to_numeric(interval_map["upper_bound_days"], errors="coerce").fillna(INFINITY_SENTINEL)
interval_map["log10_upper"] = pd.to_numeric(interval_map["log10_upper"], errors="coerce").fillna(INFINITY_SENTINEL)


def map_t_half_to_interval(t_half_series, interval_map):
    """Map each T_half_days value to the corresponding interval bin.

    Returns a DataFrame with columns:
        T_half_class_days, T_half_lower_bound, T_half_upper_bound,
        T_half_log10_lower, T_half_log10_upper

    Rows with NaN T_half_days get NaN in all output columns.
    The upper bound of the last (open-ended) bin is stored as INFINITY_SENTINEL.
    Bins are defined as [lower_bound_days, upper_bound_days).
    """
    out_cols = [
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
    ]
    result = pd.DataFrame(np.nan, index=t_half_series.index, columns=out_cols)

    for idx, val in t_half_series.items():
        if pd.isna(val):
            continue  # leaves NaN — row will be dropped before DB insert
        for _, bin_row in interval_map.iterrows():
            if bin_row["lower_bound_days"] <= val < bin_row["upper_bound_days"]:
                result.loc[idx, "T_half_class_days"] = bin_row["class_value_days"]
                result.loc[idx, "T_half_lower_bound"] = bin_row["lower_bound_days"]
                result.loc[idx, "T_half_upper_bound"] = bin_row["upper_bound_days"]  # sentinel for last bin
                result.loc[idx, "T_half_log10_lower"] = bin_row["log10_lower"]
                result.loc[idx, "T_half_log10_upper"] = bin_row["log10_upper"]  # sentinel for last bin
                break
    return result


# Apply interval mapping to all four compartment DataFrames
for df in [air_data_all, soil_data_all, water_data_all, sediment_data_all]:
    mapped = map_t_half_to_interval(df["T_half_days"], interval_map)
    for col in mapped.columns:
        df[col] = mapped[col]

# Drop rows where T_half_days is NaN (cannot be mapped to any interval)
before = {
    "air": len(air_data_all),
    "soil": len(soil_data_all),
    "water": len(water_data_all),
    "sediment": len(sediment_data_all),
}
air_data_all = air_data_all.dropna(subset=["T_half_days"])
soil_data_all = soil_data_all.dropna(subset=["T_half_days"])
water_data_all = water_data_all.dropna(subset=["T_half_days"])
sediment_data_all = sediment_data_all.dropna(subset=["T_half_days"])

for compartment, df in [
    ("air", air_data_all),
    ("soil", soil_data_all),
    ("water", water_data_all),
    ("sediment", sediment_data_all),
]:
    dropped = before[compartment] - len(df)
    print(f"{compartment}: {len(df)} rows kept, {dropped} dropped (NaN T_half_days)")

print("\nSample interval mapping (air, first 5 rows):")
print(
    air_data_all[
        [
            "reference",
            "T_half_days",
            "T_half_class_days",
            "T_half_lower_bound",
            "T_half_upper_bound",
            "T_half_log10_lower",
            "T_half_log10_upper",
        ]
    ].head()
)

air: 308 rows kept, 0 dropped (NaN T_half_days)
soil: 672 rows kept, 0 dropped (NaN T_half_days)
water: 673 rows kept, 0 dropped (NaN T_half_days)
sediment: 347 rows kept, 0 dropped (NaN T_half_days)

Sample interval mapping (air, first 5 rows):
                      reference  T_half_days  T_half_class_days  \
Compound                                                          
ETHYL CARBAMATE        air_0000     0.708300           0.708333   
PROPANE                air_0001    99.000000          70.833333   
p, p' - DDT            air_0002     7.083333           7.083333   
benzo a pyrene         air_0003     7.083333           7.083333   
dibenz(a,h)anthracene  air_0004     7.083333           7.083333   

                       T_half_lower_bound  T_half_upper_bound  \
Compound                                                        
ETHYL CARBAMATE                  0.458333            1.500000   
PROPANE                         45.833333          145.833333   
p, p' - DDT             

In [4]:
# Insert data into database using ORM
session.bulk_save_objects(
    [
        AirData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in air_data_all.iterrows()
    ]
)
session.bulk_save_objects(
    [
        SoilData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in soil_data_all.iterrows()
    ]
)
session.bulk_save_objects(
    [
        WaterData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in water_data_all.iterrows()
    ]
)
session.bulk_save_objects(
    [
        SedimentData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in sediment_data_all.iterrows()
    ]
)
session.commit()

# query AirData number of total columns
n_col = len(AirData.__table__.columns)
print(f"Number of columns in air_data: {n_col}")

Number of columns in air_data: 379


In [5]:
# test and get all data from db using SQLAlchemy ORM
air_data = pd.DataFrame(
    session.query(
        AirData.reference,
        AirData.T_half_days,
        AirData.T_half_class_days,
        AirData.T_half_lower_bound,
        AirData.T_half_upper_bound,
        AirData.T_half_log10_lower,
        AirData.T_half_log10_upper,
        AirData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)
soil_data = pd.DataFrame(
    session.query(
        SoilData.reference,
        SoilData.T_half_days,
        SoilData.T_half_class_days,
        SoilData.T_half_lower_bound,
        SoilData.T_half_upper_bound,
        SoilData.T_half_log10_lower,
        SoilData.T_half_log10_upper,
        SoilData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)
water_data = pd.DataFrame(
    session.query(
        WaterData.reference,
        WaterData.T_half_days,
        WaterData.T_half_class_days,
        WaterData.T_half_lower_bound,
        WaterData.T_half_upper_bound,
        WaterData.T_half_log10_lower,
        WaterData.T_half_log10_upper,
        WaterData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)
sediment_data = pd.DataFrame(
    session.query(
        SedimentData.reference,
        SedimentData.T_half_days,
        SedimentData.T_half_class_days,
        SedimentData.T_half_lower_bound,
        SedimentData.T_half_upper_bound,
        SedimentData.T_half_log10_lower,
        SedimentData.T_half_log10_upper,
        SedimentData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)

print(f"Air data entries in DB:      {len(air_data)}")
print(f"Soil data entries in DB:     {len(soil_data)}")
print(f"Water data entries in DB:    {len(water_data)}")
print(f"Sediment data entries in DB: {len(sediment_data)}")

print("\nExample entries from air data (interval columns included):")
print(
    air_data[
        [
            "reference",
            "T_half_days",
            "T_half_class_days",
            "T_half_lower_bound",
            "T_half_upper_bound",
            "T_half_log10_lower",
            "T_half_log10_upper",
        ]
    ].head()
)

session.close()

Air data entries in DB:      308
Soil data entries in DB:     672
Water data entries in DB:    673
Sediment data entries in DB: 347

Example entries from air data (interval columns included):
  reference  T_half_days  T_half_class_days  T_half_lower_bound  \
0  air_0000     0.708300           0.708333            0.458333   
1  air_0001    99.000000          70.833333           45.833333   
2  air_0002     7.083333           7.083333            4.687500   
3  air_0003     7.083333           7.083333            4.687500   
4  air_0004     7.083333           7.083333            4.687500   

   T_half_upper_bound  T_half_log10_lower  T_half_log10_upper  
0            1.500000              -0.339               0.176  
1          145.833333               1.661               2.164  
2           15.000000               0.671               1.176  
3           15.000000               0.671               1.176  
4           15.000000               0.671               1.176  
